In [4]:

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson

from linearmodels.panel import PanelOLS, RandomEffects
from scipy import stats

# ---------------------------------------------------------------------------
# 0. PATHS
# ---------------------------------------------------------------------------
INPUT_CSV = r"C:\Users\Downloads\1111\Excel 1.xlsx"
OUT_DIR = r"C:\Users\Downloads\New folder (4)"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# 1. LOAD RAW DATA FROM CSV
# ---------------------------------------------------------------------------
# Expected columns (header names are matched flexibly below so small
# spacing/case differences in the CSV header do not break the script):
#   Company, Industry, Year, ESG Letter Grade,
#   Total Assets (US$ bn), Total Debt (US$ bn),
#   Revenue (US$ bn), Net Income (US$ bn)

def load_input_table(path):
    """Load the manually-collected dataset regardless of whether it was saved
    as .xlsx/.xls or .csv, and regardless of the CSV's text encoding.

    The original code always called pd.read_csv(), which raises a
    UnicodeDecodeError if the file is actually an Excel workbook (Excel
    files are a binary/zip format, not text - trying to decode one as
    UTF-8 text is exactly what produced "invalid continuation byte"), or
    if a genuine CSV was saved in a non-UTF-8 encoding (Excel's "CSV
    (Comma delimited)" export on Windows is usually cp1252, not UTF-8).
    """
    ext = os.path.splitext(path)[1].lower()

    if ext in (".xlsx", ".xls", ".xlsm"):
        return pd.read_excel(path)

    if ext == ".csv":
        for encoding in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
            try:
                return pd.read_csv(path, encoding=encoding)
            except UnicodeDecodeError:
                continue
        raise UnicodeDecodeError(
            "utf-8/cp1252/latin-1", b"", 0, 1,
            f"Could not read {path} as CSV with any common encoding."
        )

    raise ValueError(
        f"Unrecognised file extension '{ext}' for INPUT_CSV. "
        "Expected .csv, .xlsx, .xls, or .xlsm."
    )


raw = load_input_table(INPUT_CSV)

# Normalise header names -> a fixed internal schema, regardless of exact
# spacing/casing/units text used in the source CSV.
def _find_col(columns, must_contain):
    """Find the first column whose lowercase name contains all tokens in must_contain."""
    for col in columns:
        low = col.lower()
        if all(tok in low for tok in must_contain):
            return col
    raise KeyError(f"Could not find a column containing {must_contain} in {list(columns)}")

col_map = {
    "Company":       _find_col(raw.columns, ["company"]),
    "Industry":      _find_col(raw.columns, ["industry"]),
    "Year":          _find_col(raw.columns, ["year"]),
    "ESG_Letter":    _find_col(raw.columns, ["esg"]),
    "TotalAssets":   _find_col(raw.columns, ["total", "asset"]),
    "TotalDebt":     _find_col(raw.columns, ["total", "debt"]),
    "Revenue":       _find_col(raw.columns, ["revenue"]),
    "NetIncome":     _find_col(raw.columns, ["net", "income"]),
}

df = raw.rename(columns={v: k for k, v in col_map.items()})[list(col_map.keys())].copy()

# Basic cleaning
df["Company"] = df["Company"].astype(str).str.strip()
df["Industry"] = df["Industry"].astype(str).str.strip()
df["ESG_Letter"] = df["ESG_Letter"].astype(str).str.strip().str.upper()
df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")

for numcol in ["TotalAssets", "TotalDebt", "Revenue", "NetIncome"]:
    df[numcol] = pd.to_numeric(df[numcol], errors="coerce")

df = df.dropna(subset=["Company", "Year"]).reset_index(drop=True)

# All values in the CSV are already US$ billions -> single common currency.
df["Currency"] = "USD"
df = df.rename(columns={"Industry": "Sector"})

# ---------------------------------------------------------------------------
# 2. ESG letter grade -> numeric score (1-10 scale, higher = better ESG)
# ---------------------------------------------------------------------------
ESG_MAP = {
    "AAA": 10, "AA": 9, "A": 8, "BBB": 7, "BB": 6, "B": 5,
    "CCC": 4, "CC": 3, "C": 2, "D": 1,
}
df["ESG_Score"] = df["ESG_Letter"].map(ESG_MAP)

unmapped = df.loc[df["ESG_Score"].isna(), "ESG_Letter"].unique()
if len(unmapped) > 0:
    print(f"WARNING: {len(unmapped)} unrecognised ESG letter grade(s) found and "
          f"left as missing (excluded from regression): {sorted(unmapped)}")

# ---------------------------------------------------------------------------
# 3. Derived variables
#    ROA        = Net Income / Total Assets
#    Firm Size  = ln(Total Assets)         [assets in $bn -> converted to $ units before log]
#    Leverage   = Total Debt / Total Assets
#    Rev Growth = (Revenue_t - Revenue_t-1) / Revenue_t-1   (per company, by year)
# ---------------------------------------------------------------------------
df = df.sort_values(["Company", "Year"]).reset_index(drop=True)

df["ROA"] = df["NetIncome"] / df["TotalAssets"]
df["FirmSize"] = np.log(df["TotalAssets"] * 1e9)   # ln(Total Assets in actual $ units)
df["Leverage"] = df["TotalDebt"] / df["TotalAssets"]
df["RevGrowth"] = df.groupby("Company")["Revenue"].pct_change(fill_method=None)

# Guard against inf/-inf: these arise from legitimate edge cases in the data
# (e.g. TotalAssets == 0 -> log(0) = -inf for FirmSize; a company reporting
# Revenue == 0 in the prior year -> division by zero = inf for RevGrowth).
# pandas' dropna() does NOT treat inf as missing, so without this step these
# rows survive dropna() and crash statsmodels/linearmodels with
# "MissingDataError: exog contains inf or nans". Converting to NaN here
# means they get excluded consistently everywhere dropna() is already used
# below (descriptive stats, correlation matrix, regressions).
derived_cols = ["ROA", "FirmSize", "Leverage", "RevGrowth"]
n_inf = np.isinf(df[derived_cols]).sum()
if n_inf.any():
    print(f"\nWARNING: replacing infinite values with NaN (excluded from analysis):")
    #print(n_inf[n_inf > 0].to_string())
    df[derived_cols] = df[derived_cols].replace([np.inf, -np.inf], np.nan)

# Drop firm-years with duplicate (Company, Year) keys, keeping the first
dupes = df.duplicated(subset=["Company", "Year"], keep="first")
if dupes.any():
    print(f"WARNING: dropping {dupes.sum()} duplicate (Company, Year) rows.")
    df = df.loc[~dupes].reset_index(drop=True)

# Panel (multi-index) version for linearmodels
panel_df = df.set_index(["Company", "Year"])

print("=" * 80)
print("DATASET BUILT:", df.shape[0], "observations x", df["Company"].nunique(), "companies")
print("Years covered:", df["Year"].min(), "-", df["Year"].max())
print("=" * 80)
df.to_csv(os.path.join(OUT_DIR, "master_dataset.csv"), index=False)

# ===========================================================================
# TABLE 4.1 - SAMPLE CHARACTERISTICS
# ===========================================================================
sample_tbl = (
    df.groupby("Company")
    .agg(Sector=("Sector", "first"),
         Currency=("Currency", "first"),
         Years=("Year", lambda x: f"{x.min()}-{x.max()}"),
         Observations=("Year", "count"))
    .reset_index()
)
sample_tbl.to_csv(os.path.join(OUT_DIR, "table_4_1_sample.csv"), index=False)
print("\nTABLE 4.1 - Sample characteristics (first 15 firms shown)")
print(sample_tbl.head(15).to_string(index=False))
print(f"... ({len(sample_tbl)} firms total, full list in table_4_1_sample.csv)")

# ===========================================================================
# TABLE 4.2 / FIGURE 4.1 - ESG TRENDS
# ===========================================================================
esg_by_year = df.pivot_table(index="Year", columns="Company", values="ESG_Score")
esg_by_year.to_csv(os.path.join(OUT_DIR, "table_4_2_esg_by_year.csv"))
print("\nTABLE 4.2 - ESG score by year (numeric, 1-10 scale) written to CSV "
      "(too wide to print with ~100 firms).")

# With ~100 firms, a per-firm legend is unreadable, so Figure 4.1 shows the
# cross-sectional distribution (mean +/- 1 SD band) by year instead of one
# line per firm.
esg_stats_by_year = df.groupby("Year")["ESG_Score"].agg(["mean", "std"])
plt.figure(figsize=(9, 6))
plt.plot(esg_stats_by_year.index, esg_stats_by_year["mean"], marker="o", color="tab:green", label="Mean ESG Score")
plt.fill_between(
    esg_stats_by_year.index,
    esg_stats_by_year["mean"] - esg_stats_by_year["std"],
    esg_stats_by_year["mean"] + esg_stats_by_year["std"],
    alpha=0.2, color="tab:green", label="+/- 1 SD",
)
plt.title("Figure 4.1 - ESG Score Trend, FTSE 100 Sample (2019-2024)")
plt.xlabel("Year")
plt.ylabel("ESG Score (1-10 scale)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fig_4_1_esg_trend.png"), dpi=150)
plt.close()

# ===========================================================================
# TABLE 4.3 / FIGURE 4.2 - ROA TRENDS
# ===========================================================================
roa_by_year = df.pivot_table(index="Year", columns="Company", values="ROA")
roa_by_year.to_csv(os.path.join(OUT_DIR, "table_4_3_roa_by_year.csv"))
print("\nTABLE 4.3 - ROA by year written to CSV (too wide to print with ~100 firms).")

roa_stats_by_year = df.groupby("Year")["ROA"].agg(["mean", "std"])
plt.figure(figsize=(9, 6))
plt.plot(roa_stats_by_year.index, roa_stats_by_year["mean"], marker="o", color="tab:blue", label="Mean ROA")
plt.fill_between(
    roa_stats_by_year.index,
    roa_stats_by_year["mean"] - roa_stats_by_year["std"],
    roa_stats_by_year["mean"] + roa_stats_by_year["std"],
    alpha=0.2, color="tab:blue", label="+/- 1 SD",
)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Figure 4.2 - ROA Trend, FTSE 100 Sample (2019-2024)")
plt.xlabel("Year")
plt.ylabel("ROA")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fig_4_2_roa_trend.png"), dpi=150)
plt.close()

# ===========================================================================
# TABLE 4.4 - DESCRIPTIVE STATISTICS
# ===========================================================================
vars5 = ["ESG_Score", "ROA", "FirmSize", "Leverage", "RevGrowth"]
desc = df[vars5].agg(["mean", "median", "std", "min", "max", "count"]).T
desc.to_csv(os.path.join(OUT_DIR, "table_4_4_descriptive_stats.csv"))
print("\nTABLE 4.4 - Descriptive statistics")
print(desc.round(4).to_string())

# ===========================================================================
# TABLE 4.5 - CORRELATION MATRIX
# ===========================================================================
corr = df[vars5].corr()
corr.to_csv(os.path.join(OUT_DIR, "table_4_5_correlation_matrix.csv"))
print("\nTABLE 4.5 - Correlation matrix")
print(corr.round(3).to_string())

# ===========================================================================
# TABLE 4.6 - PANEL REGRESSION
#   ROA = b0 + b1 ESG + b2 FirmSize + b3 Leverage + b4 RevGrowth + e
#   Reported: Pooled OLS, Fixed Effects (entity), Random Effects
# ===========================================================================
reg_df = panel_df.dropna(subset=["ROA", "ESG_Score", "FirmSize", "Leverage", "RevGrowth"]).copy()

print(f"\nRegression sample: {reg_df.shape[0]} firm-year observations "
      f"across {reg_df.index.get_level_values('Company').nunique()} firms "
      "(after dropping rows with missing values, incl. first-year RevGrowth NaNs).")

y = reg_df["ROA"]
X = reg_df[["ESG_Score", "FirmSize", "Leverage", "RevGrowth"]]
X_const = sm.add_constant(X)

# --- Pooled OLS (robust SE) ---
pooled_model = sm.OLS(y, X_const).fit(cov_type="HC1")

# --- Fixed Effects (entity/company effects) ---
# NOTE: ESG letter grades often move slowly (sometimes not at all) for a
# given firm across a short 6-year window. If a regressor has little/no
# within-firm variation, entity fixed effects can "absorb" it entirely and
# linearmodels will raise AbsorbingEffectError. We fall back to
# drop_absorbed=True in that case and flag it clearly, since it affects how
# the FE results should be interpreted (the absorbed variable's effect is
# then identified only from firms where its ESG grade did change).
from linearmodels.panel.utility import AbsorbingEffectError
try:
    fe_model = PanelOLS(y, sm.add_constant(X), entity_effects=True).fit(
        cov_type="clustered", cluster_entity=True
    )
    fe_absorbed_vars = []
except AbsorbingEffectError:
    print("\nWARNING: One or more regressors (commonly ESG_Score, since ESG "
          "grades change slowly) show little within-firm variation and were "
          "fully absorbed by the entity fixed effects. Re-fitting the Fixed "
          "Effects model with drop_absorbed=True. Interpret the FE "
          "coefficients on any dropped/absorbed variable with caution.")
    fe_model = PanelOLS(y, sm.add_constant(X), entity_effects=True, drop_absorbed=True).fit(
        cov_type="clustered", cluster_entity=True
    )
    fe_absorbed_vars = [v for v in X_const.columns if v not in fe_model.params.index]
    print(f"Variable(s) dropped from the FE model as absorbed: {fe_absorbed_vars}")

# --- Random Effects ---
re_model = RandomEffects(y, sm.add_constant(X)).fit(cov_type="clustered", cluster_entity=True)

def tidy_ols(model, name):
    out = pd.DataFrame({
        "Variable": model.params.index,
        "Coefficient": model.params.values,
        "Std.Error": model.bse.values,
        "t-stat": model.tvalues.values,
        "p-value": model.pvalues.values,
    })
    out["Model"] = name
    return out

def tidy_panel(model, name):
    out = pd.DataFrame({
        "Variable": model.params.index,
        "Coefficient": model.params.values,
        "Std.Error": model.std_errors.values,
        "t-stat": model.tstats.values,
        "p-value": model.pvalues.values,
    })
    out["Model"] = name
    return out

reg_table = pd.concat([
    tidy_ols(pooled_model, "Pooled OLS"),
    tidy_panel(fe_model, "Fixed Effects"),
    tidy_panel(re_model, "Random Effects"),
], ignore_index=True)
reg_table.to_csv(os.path.join(OUT_DIR, "table_4_6_panel_regression.csv"), index=False)

print("\nTABLE 4.6 - Panel Regression Results")
print(reg_table.round(4).to_string(index=False))

fit_stats = pd.DataFrame({
    "Model": ["Pooled OLS", "Fixed Effects", "Random Effects"],
    "R2": [pooled_model.rsquared, fe_model.rsquared, re_model.rsquared],
    "Adj_R2": [pooled_model.rsquared_adj, fe_model.rsquared, re_model.rsquared],  # linearmodels has no adj R2 attr for FE/RE by default
    "F_stat": [pooled_model.fvalue, fe_model.f_statistic.stat, re_model.f_statistic.stat],
    "F_pvalue": [pooled_model.f_pvalue, fe_model.f_statistic.pval, re_model.f_statistic.pval],
    "N_obs": [int(pooled_model.nobs), int(fe_model.nobs), int(re_model.nobs)],
})
fit_stats.to_csv(os.path.join(OUT_DIR, "table_4_6b_fit_stats.csv"), index=False)
print("\nModel fit statistics")
print(fit_stats.round(4).to_string(index=False))

# ===========================================================================
# TABLE 4.7 - DIAGNOSTIC TESTS
# ===========================================================================

# --- 4.7a Variance Inflation Factor (multicollinearity) ---
vif_data = pd.DataFrame()
vif_data["Variable"] = X_const.columns
vif_data["VIF"] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]

print("\nTABLE 4.7a - Variance Inflation Factor")
print(vif_data.round(3).to_string(index=False))

# --- 4.7b Breusch-Pagan test (heteroskedasticity), based on pooled OLS residuals ---
bp_stat, bp_pvalue, bp_fstat, bp_fpvalue = het_breuschpagan(pooled_model.resid, X_const)
bp_result = pd.DataFrame({
    "Test": ["Breusch-Pagan LM stat", "Breusch-Pagan p-value", "BP F-stat", "BP F p-value"],
    "Value": [bp_stat, bp_pvalue, bp_fstat, bp_fpvalue],
})
print("\nTABLE 4.7b - Breusch-Pagan test (heteroskedasticity)")
print(bp_result.round(4).to_string(index=False))

# --- 4.7c Durbin-Watson (autocorrelation), based on pooled OLS residuals ---
dw_stat = durbin_watson(pooled_model.resid)
print(f"\nTABLE 4.7c - Durbin-Watson statistic: {dw_stat:.4f}  (~2 = no autocorrelation)")

# --- 4.7d Hausman test (Fixed Effects vs Random Effects) ---
def hausman_test(fe, re):
    """Manual Hausman specification test comparing FE and RE estimators."""
    common_vars = [v for v in fe.params.index if v in re.params.index]
    b_fe = fe.params[common_vars]
    b_re = re.params[common_vars]
    v_fe = fe.cov.loc[common_vars, common_vars]
    v_re = re.cov.loc[common_vars, common_vars]

    diff = b_fe - b_re
    var_diff = v_fe - v_re
    # Use pseudo-inverse for numerical stability
    stat = float(diff.T @ np.linalg.pinv(var_diff.values) @ diff)
    dof = len(common_vars)
    p_value = 1 - stats.chi2.cdf(stat, dof)
    return stat, dof, p_value

h_stat, h_dof, h_pvalue = hausman_test(fe_model, re_model)
print(f"\nTABLE 4.7d - Hausman test: chi2={h_stat:.4f}, df={h_dof}, p-value={h_pvalue:.4f}")
if h_pvalue < 0.05:
    print(" -> Reject H0: Fixed Effects model is preferred.")
else:
    print(" -> Fail to reject H0: Random Effects model is preferred.")

diagnostics_summary = pd.concat([
    vif_data.rename(columns={"Value": "VIF"}),
    pd.DataFrame({"Variable": bp_result["Test"], "VIF": bp_result["Value"]}),
    pd.DataFrame({"Variable": ["Durbin-Watson"], "VIF": [dw_stat]}),
    pd.DataFrame({"Variable": ["Hausman chi2", "Hausman df", "Hausman p-value"],
                  "VIF": [h_stat, h_dof, h_pvalue]}),
], ignore_index=True)
diagnostics_summary.to_csv(os.path.join(OUT_DIR, "table_4_7_diagnostics.csv"), index=False)

print("\n" + "=" * 80)
print("All tables and figures written to:", OUT_DIR)
print("=" * 80)

C:\Users\brian\anaconda3\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)



DATASET BUILT: 600 observations x 100 companies
Years covered: 2019 - 2024

TABLE 4.1 - Sample characteristics (first 15 firms shown)
                    Company                         Sector Currency     Years  Observations
                  88 Energy                  Oil & Gas E&P      USD 2019-2024             6
                 ADM Energy                  Oil & Gas E&P      USD 2019-2024             6
                 AFC Energy          Hydrogen & Fuel Cells      USD 2019-2024             6
                     Aminex                  Oil & Gas E&P      USD 2019-2024             6
               Angus Energy                  Oil & Gas E&P      USD 2019-2024             6
 Aquila European Renewables Renewable Energy (Diversified)      USD 2019-2024             6
          Arrow Exploration                  Oil & Gas E&P      USD 2019-2024             6
           Ascent Resources                  Oil & Gas E&P      USD 2019-2024             6
       Atrato Onsite Energy       Ren

In [16]:
import os
import matplotlib.pyplot as plt

NEW_OUT_DIR = r"C:\Users\Downloads\New folder (4)"
os.makedirs(NEW_OUT_DIR, exist_ok=True)   # creates the folder if it doesn't exist yet

roa_mean_by_year = df.groupby("Year")["ROA"].mean()

plt.figure(figsize=(9, 6))
plt.plot(roa_mean_by_year.index, roa_mean_by_year.values, marker="o", color="tab:blue")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Average ROA Trend, FTSE 100 Sample (2019-2024)")
plt.xlabel("Year")
plt.ylabel("Average ROA")
plt.xticks(roa_mean_by_year.index)
plt.tight_layout()
plt.savefig(os.path.join(NEW_OUT_DIR, "fig_avg_roa_trend.png"), dpi=150)
plt.show()

C:\Users\brian\AppData\Local\Temp\ipykernel_5324\1774966403.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
import os
import numpy as np
import pandas as pd

# Assumes `df` already exists with the "Year" and "ROA" columns built earlier
# in your pipeline (i.e. run this after the derived-variables / inf-cleanup
# section of your main script).

# ---------------------------------------------------------------------------
# 1. "Average ROA Trend, FTSE 100 Sample (2019-2024)"
#    -> just the mean ROA per year
# ---------------------------------------------------------------------------
avg_roa_trend = df.groupby("Year")["ROA"].mean().round(4)

print("=" * 60)
print("Average ROA Trend, FTSE 100 Sample (2019-2024)")
print("=" * 60)
print(avg_roa_trend.to_string())

# ---------------------------------------------------------------------------
# 2. "ROA Trend, FTSE 100 Sample (2019-2024)"
#    -> mean and std ROA per year (what the +/- 1 SD band is drawn from)
# ---------------------------------------------------------------------------
roa_trend = df.groupby("Year")["ROA"].agg(["mean", "std"]).round(4)
roa_trend.columns = ["Mean ROA", "Std Dev ROA"]

print("\n" + "=" * 60)
print("ROA Trend, FTSE 100 Sample (2019-2024)")
print("=" * 60)
print(roa_trend.to_string())

Average ROA Trend, FTSE 100 Sample (2019-2024)
Year
2019    0.0269
2020    0.0275
2021    0.0239
2022   -0.6617
2023   -0.0509
2024   -0.1448

ROA Trend, FTSE 100 Sample (2019-2024)
      Mean ROA  Std Dev ROA
Year                       
2019    0.0269       0.3704
2020    0.0275       0.3668
2021    0.0239       0.3609
2022   -0.6617       6.5917
2023   -0.0509       1.1084
2024   -0.1448       0.9577


In [5]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# 1. ROA - mean vs median trend
#    Median is included because ROA's minimum (-65.4547 per Table 4.4) and
#    similar extreme loss-making firm-years pull the mean down/around,
#    making it a poor sole indicator of "typical" firm performance. The
#    median is robust to these outliers and gives a clearer sense of the
#    central tendency each year.
# ---------------------------------------------------------------------------
roa_summary_by_year = df.groupby("Year")["ROA"].agg(
    mean="mean", median="median", std="std", min="min", max="max", count="count"
)
roa_summary_by_year.to_csv(os.path.join(OUT_DIR, "table_4_3b_roa_summary_by_year.csv"))
print("\nTABLE 4.3b - ROA summary by year (mean vs median)")
print(roa_summary_by_year.round(4).to_string())

plt.figure(figsize=(9, 6))
plt.plot(roa_summary_by_year.index, roa_summary_by_year["mean"],
         marker="o", color="tab:blue", label="Mean ROA")
plt.plot(roa_summary_by_year.index, roa_summary_by_year["median"],
         marker="s", linestyle="--", color="tab:orange", label="Median ROA")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Figure 4.3 - Mean vs Median ROA, FTSE 100 Sample (2019-2024)")
plt.xlabel("Year")
plt.ylabel("ROA")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fig_4_3_roa_mean_vs_median.png"), dpi=150)
plt.close()

# ---------------------------------------------------------------------------
# 2. ROA - boxplot by year (dispersion + outliers)
# ---------------------------------------------------------------------------
years_sorted = sorted(df["Year"].dropna().unique())
roa_by_year_lists = [df.loc[df["Year"] == yr, "ROA"].dropna().values for yr in years_sorted]

plt.figure(figsize=(9, 6))
bp = plt.boxplot(
    roa_by_year_lists,
    tick_labels=[str(y) for y in years_sorted],
    showmeans=True,
    meanprops={"marker": "D", "markerfacecolor": "tab:red", "markeredgecolor": "tab:red"},
    flierprops={"marker": "o", "markersize": 4, "markerfacecolor": "gray", "alpha": 0.6},
)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Figure 4.4 - ROA Distribution by Year, FTSE 100 Sample (2019-2024)")
plt.xlabel("Year")
plt.ylabel("ROA")
plt.figtext(0.5, -0.02,
            "Box = IQR, line = median, red diamond = mean, dots = outliers",
            ha="center", fontsize=8, color="dimgray")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fig_4_4_roa_boxplot_by_year.png"), dpi=150, bbox_inches="tight")
plt.close()

# ---------------------------------------------------------------------------
# 3. ESG score - trend over the 6-year span (mean vs median)
# ---------------------------------------------------------------------------
esg_summary_by_year = df.groupby("Year")["ESG_Score"].agg(
    mean="mean", median="median", std="std", count="count"
)
esg_summary_by_year.to_csv(os.path.join(OUT_DIR, "table_4_2b_esg_summary_by_year.csv"))
print("\nTABLE 4.2b - ESG score summary by year (mean vs median)")
print(esg_summary_by_year.round(4).to_string())

plt.figure(figsize=(9, 6))
plt.plot(esg_summary_by_year.index, esg_summary_by_year["mean"],
         marker="o", color="tab:green", label="Mean ESG Score")
plt.plot(esg_summary_by_year.index, esg_summary_by_year["median"],
         marker="s", linestyle="--", color="tab:olive", label="Median ESG Score")
plt.fill_between(
    esg_summary_by_year.index,
    esg_summary_by_year["mean"] - esg_summary_by_year["std"],
    esg_summary_by_year["mean"] + esg_summary_by_year["std"],
    alpha=0.15, color="tab:green", label="+/- 1 SD (around mean)",
)
plt.title("Figure 4.5 - ESG Score Trend, Mean vs Median, FTSE 100 Sample (2019-2024)")
plt.xlabel("Year")
plt.ylabel("ESG Score (1-10 scale)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fig_4_5_esg_trend_mean_median.png"), dpi=150)
plt.close()

print("\nAdditional figures/tables written to:", OUT_DIR)


TABLE 4.3b - ROA summary by year (mean vs median)
        mean  median     std      min  max  count
Year                                             
2019  0.0269  0.0147  0.3704  -1.5387  2.5     98
2020  0.0275  0.0153  0.3668  -1.5387  2.5     99
2021  0.0239  0.0201  0.3609  -1.5387  2.5    100
2022 -0.6617  0.0265  6.5917 -65.4547  5.0    100
2023 -0.0509  0.0188  1.1084  -9.0422  5.0     99
2024 -0.1448  0.0030  0.9577  -6.0533  5.0     96

TABLE 4.2b - ESG score summary by year (mean vs median)
      mean  median     std  count
Year                             
2019  7.15     7.0  1.2583    100
2020  7.17     7.0  1.2314    100
2021  7.21     7.0  1.2085    100
2022  7.19     7.0  1.2202    100
2023  7.24     7.0  1.0929    100
2024  7.22     7.0  1.1422    100

Additional figures/tables written to: C:\Users\brian\Downloads\New folder (4)


In [1]:

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson

from linearmodels.panel import PanelOLS, RandomEffects
from scipy import stats

# ---------------------------------------------------------------------------
# 0. PATHS
# ---------------------------------------------------------------------------
INPUT_CSV = r"C:\Users\Downloads\1111\Excel 1.xlsx"
OUT_DIR = r"C:\Users\Downloads\New folder (4)"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# 1. LOAD RAW DATA FROM CSV
# ---------------------------------------------------------------------------
# Expected columns (header names are matched flexibly below so small
# spacing/case differences in the CSV header do not break the script):
#   Company, Industry, Year, ESG Letter Grade,
#   Total Assets (US$ bn), Total Debt (US$ bn),
#   Revenue (US$ bn), Net Income (US$ bn)

def load_input_table(path):
    """Load the manually-collected dataset regardless of whether it was saved
    as .xlsx/.xls or .csv, and regardless of the CSV's text encoding.

    The original code always called pd.read_csv(), which raises a
    UnicodeDecodeError if the file is actually an Excel workbook (Excel
    files are a binary/zip format, not text - trying to decode one as
    UTF-8 text is exactly what produced "invalid continuation byte"), or
    if a genuine CSV was saved in a non-UTF-8 encoding (Excel's "CSV
    (Comma delimited)" export on Windows is usually cp1252, not UTF-8).
    """
    ext = os.path.splitext(path)[1].lower()

    if ext in (".xlsx", ".xls", ".xlsm"):
        return pd.read_excel(path)

    if ext == ".csv":
        for encoding in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
            try:
                return pd.read_csv(path, encoding=encoding)
            except UnicodeDecodeError:
                continue
        raise UnicodeDecodeError(
            "utf-8/cp1252/latin-1", b"", 0, 1,
            f"Could not read {path} as CSV with any common encoding."
        )

    raise ValueError(
        f"Unrecognised file extension '{ext}' for INPUT_CSV. "
        "Expected .csv, .xlsx, .xls, or .xlsm."
    )


raw = load_input_table(INPUT_CSV)

# Normalise header names -> a fixed internal schema, regardless of exact
# spacing/casing/units text used in the source CSV.
def _find_col(columns, must_contain):
    """Find the first column whose lowercase name contains all tokens in must_contain."""
    for col in columns:
        low = col.lower()
        if all(tok in low for tok in must_contain):
            return col
    raise KeyError(f"Could not find a column containing {must_contain} in {list(columns)}")

col_map = {
    "Company":       _find_col(raw.columns, ["company"]),
    "Industry":      _find_col(raw.columns, ["industry"]),
    "Year":          _find_col(raw.columns, ["year"]),
    "ESG_Letter":    _find_col(raw.columns, ["esg"]),
    "TotalAssets":   _find_col(raw.columns, ["total", "asset"]),
    "TotalDebt":     _find_col(raw.columns, ["total", "debt"]),
    "Revenue":       _find_col(raw.columns, ["revenue"]),
    "NetIncome":     _find_col(raw.columns, ["net", "income"]),
}

df = raw.rename(columns={v: k for k, v in col_map.items()})[list(col_map.keys())].copy()

# Basic cleaning
df["Company"] = df["Company"].astype(str).str.strip()
df["Industry"] = df["Industry"].astype(str).str.strip()
df["ESG_Letter"] = df["ESG_Letter"].astype(str).str.strip().str.upper()
df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")

for numcol in ["TotalAssets", "TotalDebt", "Revenue", "NetIncome"]:
    df[numcol] = pd.to_numeric(df[numcol], errors="coerce")

df = df.dropna(subset=["Company", "Year"]).reset_index(drop=True)

# All values in the CSV are already US$ billions -> single common currency.
df["Currency"] = "USD"
df = df.rename(columns={"Industry": "Sector"})

# ---------------------------------------------------------------------------
# 2. ESG letter grade -> numeric score (1-10 scale, higher = better ESG)
# ---------------------------------------------------------------------------
ESG_MAP = {
    "AAA": 10, "AA": 9, "A": 8, "BBB": 7, "BB": 6, "B": 5,
    "CCC": 4, "CC": 3, "C": 2, "D": 1,
}
df["ESG_Score"] = df["ESG_Letter"].map(ESG_MAP)

unmapped = df.loc[df["ESG_Score"].isna(), "ESG_Letter"].unique()
if len(unmapped) > 0:
    print(f"WARNING: {len(unmapped)} unrecognised ESG letter grade(s) found and "
          f"left as missing (excluded from regression): {sorted(unmapped)}")

# ---------------------------------------------------------------------------
# 3. Derived variables
#    ROA        = Net Income / Total Assets
#    Firm Size  = ln(Total Assets)         [assets in $bn -> converted to $ units before log]
#    Leverage   = Total Debt / Total Assets
#    Rev Growth = (Revenue_t - Revenue_t-1) / Revenue_t-1   (per company, by year)
# ---------------------------------------------------------------------------
df = df.sort_values(["Company", "Year"]).reset_index(drop=True)

df["ROA"] = df["NetIncome"] / df["TotalAssets"]
df["FirmSize"] = np.log(df["TotalAssets"] * 1e9)   # ln(Total Assets in actual $ units)
df["Leverage"] = df["TotalDebt"] / df["TotalAssets"]
df["RevGrowth"] = df.groupby("Company")["Revenue"].pct_change(fill_method=None)

# Guard against inf/-inf: these arise from legitimate edge cases in the data
# (e.g. TotalAssets == 0 -> log(0) = -inf for FirmSize; a company reporting
# Revenue == 0 in the prior year -> division by zero = inf for RevGrowth).
# pandas' dropna() does NOT treat inf as missing, so without this step these
# rows survive dropna() and crash statsmodels/linearmodels with
# "MissingDataError: exog contains inf or nans". Converting to NaN here
# means they get excluded consistently everywhere dropna() is already used
# below (descriptive stats, correlation matrix, regressions).
derived_cols = ["ROA", "FirmSize", "Leverage", "RevGrowth"]
n_inf = np.isinf(df[derived_cols]).sum()
if n_inf.any():
    print(f"\nWARNING: replacing infinite values with NaN (excluded from analysis):")
    #print(n_inf[n_inf > 0].to_string())
    df[derived_cols] = df[derived_cols].replace([np.inf, -np.inf], np.nan)

# Drop firm-years with duplicate (Company, Year) keys, keeping the first
dupes = df.duplicated(subset=["Company", "Year"], keep="first")
if dupes.any():
    print(f"WARNING: dropping {dupes.sum()} duplicate (Company, Year) rows.")
    df = df.loc[~dupes].reset_index(drop=True)

# Panel (multi-index) version for linearmodels
panel_df = df.set_index(["Company", "Year"])

print("=" * 80)
print("DATASET BUILT:", df.shape[0], "observations x", df["Company"].nunique(), "companies")
print("Years covered:", df["Year"].min(), "-", df["Year"].max())
print("=" * 80)
df.to_csv(os.path.join(OUT_DIR, "master_dataset.csv"), index=False)

# ===========================================================================
# TABLE 4.1 - SAMPLE CHARACTERISTICS
# ===========================================================================
sample_tbl = (
    df.groupby("Company")
    .agg(Sector=("Sector", "first"),
         Currency=("Currency", "first"),
         Years=("Year", lambda x: f"{x.min()}-{x.max()}"),
         Observations=("Year", "count"))
    .reset_index()
)
sample_tbl.to_csv(os.path.join(OUT_DIR, "table_4_1_sample.csv"), index=False)
print("\nTABLE 4.1 - Sample characteristics (first 15 firms shown)")
print(sample_tbl.head(15).to_string(index=False))
print(f"... ({len(sample_tbl)} firms total, full list in table_4_1_sample.csv)")

# ===========================================================================
# TABLE 4.2 / FIGURE 4.1 - ESG TRENDS
# ===========================================================================
esg_by_year = df.pivot_table(index="Year", columns="Company", values="ESG_Score")
esg_by_year.to_csv(os.path.join(OUT_DIR, "table_4_2_esg_by_year.csv"))
print("\nTABLE 4.2 - ESG score by year (numeric, 1-10 scale) written to CSV "
      "(too wide to print with ~100 firms).")

# With ~100 firms, a per-firm legend is unreadable, so Figure 4.1 shows the
# cross-sectional distribution (mean +/- 1 SD band) by year instead of one
# line per firm.
esg_stats_by_year = df.groupby("Year")["ESG_Score"].agg(["mean", "std"])
plt.figure(figsize=(9, 6))
plt.plot(esg_stats_by_year.index, esg_stats_by_year["mean"], marker="o", color="tab:green", label="Mean ESG Score")
plt.fill_between(
    esg_stats_by_year.index,
    esg_stats_by_year["mean"] - esg_stats_by_year["std"],
    esg_stats_by_year["mean"] + esg_stats_by_year["std"],
    alpha=0.2, color="tab:green", label="+/- 1 SD",
)
plt.title("Figure 4.1 - ESG Score Trend, FTSE 100 Sample (2019-2024)")
plt.xlabel("Year")
plt.ylabel("ESG Score (1-10 scale)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fig_4_1_esg_trend.png"), dpi=150)
plt.close()

# ===========================================================================
# TABLE 4.3 / FIGURE 4.2 - ROA TRENDS
# ===========================================================================
roa_by_year = df.pivot_table(index="Year", columns="Company", values="ROA")
roa_by_year.to_csv(os.path.join(OUT_DIR, "table_4_3_roa_by_year.csv"))
print("\nTABLE 4.3 - ROA by year written to CSV (too wide to print with ~100 firms).")

roa_stats_by_year = df.groupby("Year")["ROA"].agg(["mean", "std"])
plt.figure(figsize=(9, 6))
plt.plot(roa_stats_by_year.index, roa_stats_by_year["mean"], marker="o", color="tab:blue", label="Mean ROA")
plt.fill_between(
    roa_stats_by_year.index,
    roa_stats_by_year["mean"] - roa_stats_by_year["std"],
    roa_stats_by_year["mean"] + roa_stats_by_year["std"],
    alpha=0.2, color="tab:blue", label="+/- 1 SD",
)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Figure 4.2 - ROA Trend, FTSE 100 Sample (2019-2024)")
plt.xlabel("Year")
plt.ylabel("ROA")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fig_4_2_roa_trend.png"), dpi=150)
plt.close()

# ===========================================================================
# TABLE 4.4 - DESCRIPTIVE STATISTICS
# ===========================================================================
vars5 = ["ESG_Score", "ROA", "FirmSize", "Leverage", "RevGrowth"]
desc = df[vars5].agg(["mean", "median", "std", "min", "max", "count"]).T
desc.to_csv(os.path.join(OUT_DIR, "table_4_4_descriptive_stats.csv"))
print("\nTABLE 4.4 - Descriptive statistics")
print(desc.round(4).to_string())

# ===========================================================================
# TABLE 4.5 - CORRELATION MATRIX
# ===========================================================================
corr = df[vars5].corr()
corr.to_csv(os.path.join(OUT_DIR, "table_4_5_correlation_matrix.csv"))
print("\nTABLE 4.5 - Correlation matrix")
print(corr.round(3).to_string())

# ===========================================================================
# TABLE 4.6 - PANEL REGRESSION
#   ROA = b0 + b1 ESG + b2 FirmSize + b3 Leverage + b4 RevGrowth + e
#   Reported: Pooled OLS, Fixed Effects (entity), Random Effects
# ===========================================================================
reg_df = panel_df.dropna(subset=["ROA", "ESG_Score", "FirmSize", "Leverage", "RevGrowth"]).copy()

print(f"\nRegression sample: {reg_df.shape[0]} firm-year observations "
      f"across {reg_df.index.get_level_values('Company').nunique()} firms "
      "(after dropping rows with missing values, incl. first-year RevGrowth NaNs).")

y = reg_df["ROA"]
X = reg_df[["ESG_Score", "FirmSize", "Leverage", "RevGrowth"]]
X_const = sm.add_constant(X)

# --- Pooled OLS (robust SE) ---
pooled_model = sm.OLS(y, X_const).fit(cov_type="HC1")

# --- Fixed Effects (entity/company effects) ---
# NOTE: ESG letter grades often move slowly (sometimes not at all) for a
# given firm across a short 6-year window. If a regressor has little/no
# within-firm variation, entity fixed effects can "absorb" it entirely and
# linearmodels will raise AbsorbingEffectError. We fall back to
# drop_absorbed=True in that case and flag it clearly, since it affects how
# the FE results should be interpreted (the absorbed variable's effect is
# then identified only from firms where its ESG grade did change).
from linearmodels.panel.utility import AbsorbingEffectError
try:
    fe_model = PanelOLS(y, sm.add_constant(X), entity_effects=True).fit(
        cov_type="clustered", cluster_entity=True
    )
    fe_absorbed_vars = []
except AbsorbingEffectError:
    print("\nWARNING: One or more regressors (commonly ESG_Score, since ESG "
          "grades change slowly) show little within-firm variation and were "
          "fully absorbed by the entity fixed effects. Re-fitting the Fixed "
          "Effects model with drop_absorbed=True. Interpret the FE "
          "coefficients on any dropped/absorbed variable with caution.")
    fe_model = PanelOLS(y, sm.add_constant(X), entity_effects=True, drop_absorbed=True).fit(
        cov_type="clustered", cluster_entity=True
    )
    fe_absorbed_vars = [v for v in X_const.columns if v not in fe_model.params.index]
    print(f"Variable(s) dropped from the FE model as absorbed: {fe_absorbed_vars}")

# --- Random Effects (clustered by firm/entity) ---
re_model = RandomEffects(y, sm.add_constant(X)).fit(cov_type="clustered", cluster_entity=True)

def tidy_ols(model, name):
    out = pd.DataFrame({
        "Variable": model.params.index,
        "Coefficient": model.params.values,
        "Std.Error": model.bse.values,
        "t-stat": model.tvalues.values,
        "p-value": model.pvalues.values,
    })
    out["Model"] = name
    return out

def tidy_panel(model, name):
    out = pd.DataFrame({
        "Variable": model.params.index,
        "Coefficient": model.params.values,
        "Std.Error": model.std_errors.values,
        "t-stat": model.tstats.values,
        "p-value": model.pvalues.values,
    })
    out["Model"] = name
    return out

reg_table = pd.concat([
    tidy_ols(pooled_model, "Pooled OLS"),
    tidy_panel(fe_model, "Fixed Effects"),
    tidy_panel(re_model, "Random Effects"),
], ignore_index=True)
reg_table.to_csv(os.path.join(OUT_DIR, "table_4_6_panel_regression.csv"), index=False)

print("\nTABLE 4.6 - Panel Regression Results")
print(reg_table.round(4).to_string(index=False))

fit_stats = pd.DataFrame({
    "Model": ["Pooled OLS", "Fixed Effects", "Random Effects"],
    "R2": [pooled_model.rsquared, fe_model.rsquared, re_model.rsquared],
    "Adj_R2": [pooled_model.rsquared_adj, fe_model.rsquared, re_model.rsquared],  # linearmodels has no adj R2 attr for FE/RE by default
    "F_stat": [pooled_model.fvalue, fe_model.f_statistic.stat, re_model.f_statistic.stat],
    "F_pvalue": [pooled_model.f_pvalue, fe_model.f_statistic.pval, re_model.f_statistic.pval],
    "N_obs": [int(pooled_model.nobs), int(fe_model.nobs), int(re_model.nobs)],
})
fit_stats.to_csv(os.path.join(OUT_DIR, "table_4_6b_fit_stats.csv"), index=False)
print("\nModel fit statistics")
print(fit_stats.round(4).to_string(index=False))

# ===========================================================================
# TABLE 4.6c - RANDOM EFFECTS RESULTS WITH CLUSTERED (BY-FIRM) STANDARD ERRORS
#   Pulls the ESG, FirmSize, Leverage and RevGrowth coefficients out of
#   re_model (already fit above with cov_type="clustered", cluster_entity=True)
#   into a standalone table, formatted for direct use in the write-up.
# ===========================================================================
vars_of_interest = ["ESG_Score", "FirmSize", "Leverage", "RevGrowth"]

re_clustered_tbl = pd.DataFrame({
    "Variable": re_model.params.index,
    "Coefficient": re_model.params.values,
    "Clustered_Std_Error": re_model.std_errors.values,
    "t_stat": re_model.tstats.values,
    "p_value": re_model.pvalues.values,
})

# Order rows: intercept first, then the four predictors of interest
re_clustered_tbl = re_clustered_tbl.set_index("Variable").loc[
    ["const"] + vars_of_interest
].reset_index()

def sig_stars(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "*"
    return ""

re_clustered_tbl["Sig"] = re_clustered_tbl["p_value"].apply(sig_stars)

re_clustered_tbl.to_csv(
    os.path.join(OUT_DIR, "table_4_6c_RE_clustered_results.csv"), index=False
)

print("\nTABLE 4.6c - Random Effects Model, Clustered (by-firm) Standard Errors")
print(re_clustered_tbl.round(4).to_string(index=False))

print("\n--- Formatted for write-up (paste into results section) ---")
for _, row in re_clustered_tbl.iterrows():
    if row["Variable"] == "const":
        continue
    print(
        f"{row['Variable']}: b = {row['Coefficient']:.4f}, "
        f"SE = {row['Clustered_Std_Error']:.4f}, "
        f"t = {row['t_stat']:.3f}, "
        f"p = {row['p_value']:.4f}{row['Sig']}"
    )

# ===========================================================================
# TABLE 4.7 - DIAGNOSTIC TESTS
# ===========================================================================

# --- 4.7a Variance Inflation Factor (multicollinearity) ---
vif_data = pd.DataFrame()
vif_data["Variable"] = X_const.columns
vif_data["VIF"] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]

print("\nTABLE 4.7a - Variance Inflation Factor")
print(vif_data.round(3).to_string(index=False))

# --- 4.7b Breusch-Pagan test (heteroskedasticity), based on pooled OLS residuals ---
bp_stat, bp_pvalue, bp_fstat, bp_fpvalue = het_breuschpagan(pooled_model.resid, X_const)
bp_result = pd.DataFrame({
    "Test": ["Breusch-Pagan LM stat", "Breusch-Pagan p-value", "BP F-stat", "BP F p-value"],
    "Value": [bp_stat, bp_pvalue, bp_fstat, bp_fpvalue],
})
print("\nTABLE 4.7b - Breusch-Pagan test (heteroskedasticity)")
print(bp_result.round(4).to_string(index=False))

# --- 4.7c Durbin-Watson (autocorrelation), based on pooled OLS residuals ---
dw_stat = durbin_watson(pooled_model.resid)
print(f"\nTABLE 4.7c - Durbin-Watson statistic: {dw_stat:.4f}  (~2 = no autocorrelation)")

# --- 4.7d Hausman test (Fixed Effects vs Random Effects) ---
def hausman_test(fe, re):
    """Manual Hausman specification test comparing FE and RE estimators."""
    common_vars = [v for v in fe.params.index if v in re.params.index]
    b_fe = fe.params[common_vars]
    b_re = re.params[common_vars]
    v_fe = fe.cov.loc[common_vars, common_vars]
    v_re = re.cov.loc[common_vars, common_vars]

    diff = b_fe - b_re
    var_diff = v_fe - v_re
    # Use pseudo-inverse for numerical stability
    stat = float(diff.T @ np.linalg.pinv(var_diff.values) @ diff)
    dof = len(common_vars)
    p_value = 1 - stats.chi2.cdf(stat, dof)
    return stat, dof, p_value

h_stat, h_dof, h_pvalue = hausman_test(fe_model, re_model)
print(f"\nTABLE 4.7d - Hausman test: chi2={h_stat:.4f}, df={h_dof}, p-value={h_pvalue:.4f}")
if h_pvalue < 0.05:
    print(" -> Reject H0: Fixed Effects model is preferred.")
else:
    print(" -> Fail to reject H0: Random Effects model is preferred.")

diagnostics_summary = pd.concat([
    vif_data.rename(columns={"Value": "VIF"}),
    pd.DataFrame({"Variable": bp_result["Test"], "VIF": bp_result["Value"]}),
    pd.DataFrame({"Variable": ["Durbin-Watson"], "VIF": [dw_stat]}),
    pd.DataFrame({"Variable": ["Hausman chi2", "Hausman df", "Hausman p-value"],
                  "VIF": [h_stat, h_dof, h_pvalue]}),
], ignore_index=True)
diagnostics_summary.to_csv(os.path.join(OUT_DIR, "table_4_7_diagnostics.csv"), index=False)

print("\n" + "=" * 80)
print("All tables and figures written to:", OUT_DIR)
print("=" * 80)

C:\Users\brian\anaconda3\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)



DATASET BUILT: 600 observations x 100 companies
Years covered: 2019 - 2024

TABLE 4.1 - Sample characteristics (first 15 firms shown)
                    Company                         Sector Currency     Years  Observations
                  88 Energy                  Oil & Gas E&P      USD 2019-2024             6
                 ADM Energy                  Oil & Gas E&P      USD 2019-2024             6
                 AFC Energy          Hydrogen & Fuel Cells      USD 2019-2024             6
                     Aminex                  Oil & Gas E&P      USD 2019-2024             6
               Angus Energy                  Oil & Gas E&P      USD 2019-2024             6
 Aquila European Renewables Renewable Energy (Diversified)      USD 2019-2024             6
          Arrow Exploration                  Oil & Gas E&P      USD 2019-2024             6
           Ascent Resources                  Oil & Gas E&P      USD 2019-2024             6
       Atrato Onsite Energy       Ren

In [2]:

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan

from linearmodels.panel import PanelOLS, RandomEffects
from scipy import stats

# ---------------------------------------------------------------------------
# 0. PATHS
# ---------------------------------------------------------------------------
INPUT_CSV = r"C:\Users\Downloads\1111\Excel 1.xlsx"
OUT_DIR = r"C:\Users\Downloads\New folder (4)"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------------------------------------------------------------------
# 1. LOAD RAW DATA FROM CSV
# ---------------------------------------------------------------------------
# Expected columns (header names are matched flexibly below so small
# spacing/case differences in the CSV header do not break the script):
#   Company, Industry, Year, ESG Letter Grade,
#   Total Assets (US$ bn), Total Debt (US$ bn),
#   Revenue (US$ bn), Net Income (US$ bn)

def load_input_table(path):
    """Load the manually-collected dataset regardless of whether it was saved
    as .xlsx/.xls or .csv, and regardless of the CSV's text encoding.

    The original code always called pd.read_csv(), which raises a
    UnicodeDecodeError if the file is actually an Excel workbook (Excel
    files are a binary/zip format, not text - trying to decode one as
    UTF-8 text is exactly what produced "invalid continuation byte"), or
    if a genuine CSV was saved in a non-UTF-8 encoding (Excel's "CSV
    (Comma delimited)" export on Windows is usually cp1252, not UTF-8).
    """
    ext = os.path.splitext(path)[1].lower()

    if ext in (".xlsx", ".xls", ".xlsm"):
        return pd.read_excel(path)

    if ext == ".csv":
        for encoding in ("utf-8", "utf-8-sig", "cp1252", "latin-1"):
            try:
                return pd.read_csv(path, encoding=encoding)
            except UnicodeDecodeError:
                continue
        raise UnicodeDecodeError(
            "utf-8/cp1252/latin-1", b"", 0, 1,
            f"Could not read {path} as CSV with any common encoding."
        )

    raise ValueError(
        f"Unrecognised file extension '{ext}' for INPUT_CSV. "
        "Expected .csv, .xlsx, .xls, or .xlsm."
    )


raw = load_input_table(INPUT_CSV)

# Normalise header names -> a fixed internal schema, regardless of exact
# spacing/casing/units text used in the source CSV.
def _find_col(columns, must_contain):
    """Find the first column whose lowercase name contains all tokens in must_contain."""
    for col in columns:
        low = col.lower()
        if all(tok in low for tok in must_contain):
            return col
    raise KeyError(f"Could not find a column containing {must_contain} in {list(columns)}")

col_map = {
    "Company":       _find_col(raw.columns, ["company"]),
    "Industry":      _find_col(raw.columns, ["industry"]),
    "Year":          _find_col(raw.columns, ["year"]),
    "ESG_Letter":    _find_col(raw.columns, ["esg"]),
    "TotalAssets":   _find_col(raw.columns, ["total", "asset"]),
    "TotalDebt":     _find_col(raw.columns, ["total", "debt"]),
    "Revenue":       _find_col(raw.columns, ["revenue"]),
    "NetIncome":     _find_col(raw.columns, ["net", "income"]),
}

df = raw.rename(columns={v: k for k, v in col_map.items()})[list(col_map.keys())].copy()

# Basic cleaning
df["Company"] = df["Company"].astype(str).str.strip()
df["Industry"] = df["Industry"].astype(str).str.strip()
df["ESG_Letter"] = df["ESG_Letter"].astype(str).str.strip().str.upper()
df["Year"] = pd.to_numeric(df["Year"], errors="coerce").astype("Int64")

for numcol in ["TotalAssets", "TotalDebt", "Revenue", "NetIncome"]:
    df[numcol] = pd.to_numeric(df[numcol], errors="coerce")

df = df.dropna(subset=["Company", "Year"]).reset_index(drop=True)

# All values in the CSV are already US$ billions -> single common currency.
df["Currency"] = "USD"
df = df.rename(columns={"Industry": "Sector"})

# ---------------------------------------------------------------------------
# 2. ESG letter grade -> numeric score (1-10 scale, higher = better ESG)
# ---------------------------------------------------------------------------
ESG_MAP = {
    "AAA": 10, "AA": 9, "A": 8, "BBB": 7, "BB": 6, "B": 5,
    "CCC": 4, "CC": 3, "C": 2, "D": 1,
}
df["ESG_Score"] = df["ESG_Letter"].map(ESG_MAP)

unmapped = df.loc[df["ESG_Score"].isna(), "ESG_Letter"].unique()
if len(unmapped) > 0:
    print(f"WARNING: {len(unmapped)} unrecognised ESG letter grade(s) found and "
          f"left as missing (excluded from regression): {sorted(unmapped)}")

# ---------------------------------------------------------------------------
# 3. Derived variables
#    ROA        = Net Income / Total Assets
#    Firm Size  = ln(Total Assets)         [assets in $bn -> converted to $ units before log]
#    Leverage   = Total Debt / Total Assets
#    Rev Growth = (Revenue_t - Revenue_t-1) / Revenue_t-1   (per company, by year)
# ---------------------------------------------------------------------------
df = df.sort_values(["Company", "Year"]).reset_index(drop=True)

df["ROA"] = df["NetIncome"] / df["TotalAssets"]
df["FirmSize"] = np.log(df["TotalAssets"] * 1e9)   # ln(Total Assets in actual $ units)
df["Leverage"] = df["TotalDebt"] / df["TotalAssets"]
df["RevGrowth"] = df.groupby("Company")["Revenue"].pct_change(fill_method=None)

# Guard against inf/-inf: these arise from legitimate edge cases in the data
# (e.g. TotalAssets == 0 -> log(0) = -inf for FirmSize; a company reporting
# Revenue == 0 in the prior year -> division by zero = inf for RevGrowth).
# pandas' dropna() does NOT treat inf as missing, so without this step these
# rows survive dropna() and crash statsmodels/linearmodels with
# "MissingDataError: exog contains inf or nans". Converting to NaN here
# means they get excluded consistently everywhere dropna() is already used
# below (descriptive stats, correlation matrix, regressions).
derived_cols = ["ROA", "FirmSize", "Leverage", "RevGrowth"]
n_inf = np.isinf(df[derived_cols]).sum()
if n_inf.any():
    print(f"\nWARNING: replacing infinite values with NaN (excluded from analysis):")
    #print(n_inf[n_inf > 0].to_string())
    df[derived_cols] = df[derived_cols].replace([np.inf, -np.inf], np.nan)

# Drop firm-years with duplicate (Company, Year) keys, keeping the first
dupes = df.duplicated(subset=["Company", "Year"], keep="first")
if dupes.any():
    print(f"WARNING: dropping {dupes.sum()} duplicate (Company, Year) rows.")
    df = df.loc[~dupes].reset_index(drop=True)

# Panel (multi-index) version for linearmodels
panel_df = df.set_index(["Company", "Year"])

print("=" * 80)
print("DATASET BUILT:", df.shape[0], "observations x", df["Company"].nunique(), "companies")
print("Years covered:", df["Year"].min(), "-", df["Year"].max())
print("=" * 80)
df.to_csv(os.path.join(OUT_DIR, "master_dataset.csv"), index=False)

# ===========================================================================
# TABLE 4.1 - SAMPLE CHARACTERISTICS
# ===========================================================================
sample_tbl = (
    df.groupby("Company")
    .agg(Sector=("Sector", "first"),
         Currency=("Currency", "first"),
         Years=("Year", lambda x: f"{x.min()}-{x.max()}"),
         Observations=("Year", "count"))
    .reset_index()
)
sample_tbl.to_csv(os.path.join(OUT_DIR, "table_4_1_sample.csv"), index=False)
print("\nTABLE 4.1 - Sample characteristics (first 15 firms shown)")
print(sample_tbl.head(15).to_string(index=False))
print(f"... ({len(sample_tbl)} firms total, full list in table_4_1_sample.csv)")

# ===========================================================================
# TABLE 4.2 / FIGURE 4.1 - ESG TRENDS
# ===========================================================================
esg_by_year = df.pivot_table(index="Year", columns="Company", values="ESG_Score")
esg_by_year.to_csv(os.path.join(OUT_DIR, "table_4_2_esg_by_year.csv"))
print("\nTABLE 4.2 - ESG score by year (numeric, 1-10 scale) written to CSV "
      "(too wide to print with ~100 firms).")

# With ~100 firms, a per-firm legend is unreadable, so Figure 4.1 shows the
# cross-sectional distribution (mean +/- 1 SD band) by year instead of one
# line per firm.
esg_stats_by_year = df.groupby("Year")["ESG_Score"].agg(["mean", "std"])
plt.figure(figsize=(9, 6))
plt.plot(esg_stats_by_year.index, esg_stats_by_year["mean"], marker="o", color="tab:green", label="Mean ESG Score")
plt.fill_between(
    esg_stats_by_year.index,
    esg_stats_by_year["mean"] - esg_stats_by_year["std"],
    esg_stats_by_year["mean"] + esg_stats_by_year["std"],
    alpha=0.2, color="tab:green", label="+/- 1 SD",
)
plt.title("Figure 4.1 - ESG Score Trend, FTSE 100 Sample (2019-2024)")
plt.xlabel("Year")
plt.ylabel("ESG Score (1-10 scale)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fig_4_1_esg_trend.png"), dpi=150)
plt.close()

# ===========================================================================
# TABLE 4.3 / FIGURE 4.2 - ROA TRENDS
# ===========================================================================
roa_by_year = df.pivot_table(index="Year", columns="Company", values="ROA")
roa_by_year.to_csv(os.path.join(OUT_DIR, "table_4_3_roa_by_year.csv"))
print("\nTABLE 4.3 - ROA by year written to CSV (too wide to print with ~100 firms).")

roa_stats_by_year = df.groupby("Year")["ROA"].agg(["mean", "std"])
plt.figure(figsize=(9, 6))
plt.plot(roa_stats_by_year.index, roa_stats_by_year["mean"], marker="o", color="tab:blue", label="Mean ROA")
plt.fill_between(
    roa_stats_by_year.index,
    roa_stats_by_year["mean"] - roa_stats_by_year["std"],
    roa_stats_by_year["mean"] + roa_stats_by_year["std"],
    alpha=0.2, color="tab:blue", label="+/- 1 SD",
)
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Figure 4.2 - ROA Trend, FTSE 100 Sample (2019-2024)")
plt.xlabel("Year")
plt.ylabel("ROA")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "fig_4_2_roa_trend.png"), dpi=150)
plt.close()

# ===========================================================================
# TABLE 4.4 - DESCRIPTIVE STATISTICS
# ===========================================================================
vars5 = ["ESG_Score", "ROA", "FirmSize", "Leverage", "RevGrowth"]
desc = df[vars5].agg(["mean", "median", "std", "min", "max", "count"]).T
desc.to_csv(os.path.join(OUT_DIR, "table_4_4_descriptive_stats.csv"))
print("\nTABLE 4.4 - Descriptive statistics")
print(desc.round(4).to_string())

# ===========================================================================
# TABLE 4.5 - CORRELATION MATRIX
# ===========================================================================
corr = df[vars5].corr()
corr.to_csv(os.path.join(OUT_DIR, "table_4_5_correlation_matrix.csv"))
print("\nTABLE 4.5 - Correlation matrix")
print(corr.round(3).to_string())

# ===========================================================================
# TABLE 4.6 - PANEL REGRESSION
#   ROA = b0 + b1 ESG + b2 FirmSize + b3 Leverage + b4 RevGrowth + e
#   Reported: Pooled OLS, Fixed Effects (entity), Random Effects
# ===========================================================================
reg_df = panel_df.dropna(subset=["ROA", "ESG_Score", "FirmSize", "Leverage", "RevGrowth"]).copy()

print(f"\nRegression sample: {reg_df.shape[0]} firm-year observations "
      f"across {reg_df.index.get_level_values('Company').nunique()} firms "
      "(after dropping rows with missing values, incl. first-year RevGrowth NaNs).")

y = reg_df["ROA"]
X = reg_df[["ESG_Score", "FirmSize", "Leverage", "RevGrowth"]]
X_const = sm.add_constant(X)

# --- Pooled OLS (robust SE) ---
pooled_model = sm.OLS(y, X_const).fit(cov_type="HC1")

# --- Fixed Effects (entity/company effects) ---
# NOTE: ESG letter grades often move slowly (sometimes not at all) for a
# given firm across a short 6-year window. If a regressor has little/no
# within-firm variation, entity fixed effects can "absorb" it entirely and
# linearmodels will raise AbsorbingEffectError. We fall back to
# drop_absorbed=True in that case and flag it clearly, since it affects how
# the FE results should be interpreted (the absorbed variable's effect is
# then identified only from firms where its ESG grade did change).
from linearmodels.panel.utility import AbsorbingEffectError
try:
    fe_model = PanelOLS(y, sm.add_constant(X), entity_effects=True).fit(
        cov_type="clustered", cluster_entity=True
    )
    fe_absorbed_vars = []
except AbsorbingEffectError:
    print("\nWARNING: One or more regressors (commonly ESG_Score, since ESG "
          "grades change slowly) show little within-firm variation and were "
          "fully absorbed by the entity fixed effects. Re-fitting the Fixed "
          "Effects model with drop_absorbed=True. Interpret the FE "
          "coefficients on any dropped/absorbed variable with caution.")
    fe_model = PanelOLS(y, sm.add_constant(X), entity_effects=True, drop_absorbed=True).fit(
        cov_type="clustered", cluster_entity=True
    )
    fe_absorbed_vars = [v for v in X_const.columns if v not in fe_model.params.index]
    print(f"Variable(s) dropped from the FE model as absorbed: {fe_absorbed_vars}")

# --- Random Effects (clustered by firm/entity) ---
re_model = RandomEffects(y, sm.add_constant(X)).fit(cov_type="clustered", cluster_entity=True)

def tidy_ols(model, name):
    out = pd.DataFrame({
        "Variable": model.params.index,
        "Coefficient": model.params.values,
        "Std.Error": model.bse.values,
        "t-stat": model.tvalues.values,
        "p-value": model.pvalues.values,
    })
    out["Model"] = name
    return out

def tidy_panel(model, name):
    out = pd.DataFrame({
        "Variable": model.params.index,
        "Coefficient": model.params.values,
        "Std.Error": model.std_errors.values,
        "t-stat": model.tstats.values,
        "p-value": model.pvalues.values,
    })
    out["Model"] = name
    return out

reg_table = pd.concat([
    tidy_ols(pooled_model, "Pooled OLS"),
    tidy_panel(fe_model, "Fixed Effects"),
    tidy_panel(re_model, "Random Effects"),
], ignore_index=True)
reg_table.to_csv(os.path.join(OUT_DIR, "table_4_6_panel_regression.csv"), index=False)

print("\nTABLE 4.6 - Panel Regression Results")
print(reg_table.round(4).to_string(index=False))

fit_stats = pd.DataFrame({
    "Model": ["Pooled OLS", "Fixed Effects", "Random Effects"],
    "R2": [pooled_model.rsquared, fe_model.rsquared, re_model.rsquared],
    "Adj_R2": [pooled_model.rsquared_adj, fe_model.rsquared, re_model.rsquared],  # linearmodels has no adj R2 attr for FE/RE by default
    "F_stat": [pooled_model.fvalue, fe_model.f_statistic.stat, re_model.f_statistic.stat],
    "F_pvalue": [pooled_model.f_pvalue, fe_model.f_statistic.pval, re_model.f_statistic.pval],
    "N_obs": [int(pooled_model.nobs), int(fe_model.nobs), int(re_model.nobs)],
})
fit_stats.to_csv(os.path.join(OUT_DIR, "table_4_6b_fit_stats.csv"), index=False)
print("\nModel fit statistics")
print(fit_stats.round(4).to_string(index=False))

# ===========================================================================
# TABLE 4.6c - RANDOM EFFECTS RESULTS WITH CLUSTERED (BY-FIRM) STANDARD ERRORS
#   Pulls the ESG, FirmSize, Leverage and RevGrowth coefficients out of
#   re_model (already fit above with cov_type="clustered", cluster_entity=True)
#   into a standalone table, formatted for direct use in the write-up.
# ===========================================================================
vars_of_interest = ["ESG_Score", "FirmSize", "Leverage", "RevGrowth"]

re_clustered_tbl = pd.DataFrame({
    "Variable": re_model.params.index,
    "Coefficient": re_model.params.values,
    "Clustered_Std_Error": re_model.std_errors.values,
    "t_stat": re_model.tstats.values,
    "p_value": re_model.pvalues.values,
})

# Order rows: intercept first, then the four predictors of interest
re_clustered_tbl = re_clustered_tbl.set_index("Variable").loc[
    ["const"] + vars_of_interest
].reset_index()

def sig_stars(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "*"
    return ""

re_clustered_tbl["Sig"] = re_clustered_tbl["p_value"].apply(sig_stars)

re_clustered_tbl.to_csv(
    os.path.join(OUT_DIR, "table_4_6c_RE_clustered_results.csv"), index=False
)

print("\nTABLE 4.6c - Random Effects Model, Clustered (by-firm) Standard Errors")
print(re_clustered_tbl.round(4).to_string(index=False))

print("\n--- Formatted for write-up (paste into results section) ---")
for _, row in re_clustered_tbl.iterrows():
    if row["Variable"] == "const":
        continue
    print(
        f"{row['Variable']}: b = {row['Coefficient']:.4f}, "
        f"SE = {row['Clustered_Std_Error']:.4f}, "
        f"t = {row['t_stat']:.3f}, "
        f"p = {row['p_value']:.4f}{row['Sig']}"
    )

# ===========================================================================
# TABLE 4.7 - DIAGNOSTIC TESTS
#
#   4.7a  Variance Inflation Factor            - multicollinearity
#   4.7b  Breusch-Pagan test (White-style)      - H0: homoskedastic errors
#   4.7c  Breusch-Pagan LM test for panel effects
#         (Breusch & Pagan, 1980)               - H0: no panel/random effects
#                                                  (Var(u_i) = 0, i.e. pooled
#                                                  OLS is adequate; a firm-year
#                                                  observation's errors are
#                                                  not correlated with other
#                                                  years for the same firm)
#   4.7d  Wooldridge test for serial correlation
#         in panel data (Wooldridge, 2002;
#         Drukker, 2003)                        - H0: no first-order
#                                                  autocorrelation in the
#                                                  idiosyncratic error term
#   4.7e  Hausman test                          - FE vs RE specification
#
#   NOTE: 4.7b and 4.7c are BOTH "Breusch-Pagan" tests but they are testing
#   two completely different null hypotheses and must not be conflated:
#     - 4.7b (heteroskedasticity) asks whether the VARIANCE of the residuals
#       is constant across observations.
#     - 4.7c (LM test for panel effects) asks whether there is significant
#       FIRM-LEVEL (entity) variance component in the composite error term,
#       i.e. whether a panel estimator (RE/FE) is needed at all instead of
#       pooled OLS.
#   They are reported as separate tables below for that reason.
#
#   NOTE ON DURBIN-WATSON: the classic Durbin-Watson statistic assumes a
#   single time-ordered series and is not a valid test for serial
#   correlation in panel (firm x year) data, because it does not account for
#   the panel structure (it would treat consecutive rows from DIFFERENT
#   firms as though they were consecutive time periods for the same firm
#   unless the panel is perfectly balanced and sorted, and even then it does
#   not correct standard errors/degrees of freedom for the panel setting).
#   It has therefore been replaced below with the Wooldridge (2002) test for
#   serial correlation in panel data, which is the standard approach in
#   applied panel-data work (e.g. Drukker, 2003; implemented in Stata as
#   `xtserial`).
# ===========================================================================

# --- 4.7a Variance Inflation Factor (multicollinearity) ---
vif_data = pd.DataFrame()
vif_data["Variable"] = X_const.columns
vif_data["VIF"] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]

print("\nTABLE 4.7a - Variance Inflation Factor")
print(vif_data.round(3).to_string(index=False))

# --- 4.7b Breusch-Pagan test for HETEROSKEDASTICITY -----------------------
# H0: error variance is constant (homoskedastic) across observations.
# Based on pooled OLS residuals regressed on the model's regressors.
bp_het_stat, bp_het_pvalue, bp_het_fstat, bp_het_fpvalue = het_breuschpagan(pooled_model.resid, X_const)
bp_het_result = pd.DataFrame({
    "Test": ["BP (heteroskedasticity) LM stat", "BP (heteroskedasticity) p-value",
              "BP (heteroskedasticity) F-stat", "BP (heteroskedasticity) F p-value"],
    "Value": [bp_het_stat, bp_het_pvalue, bp_het_fstat, bp_het_fpvalue],
})
print("\nTABLE 4.7b - Breusch-Pagan test for HETEROSKEDASTICITY")
print("H0: error variance is constant across observations (homoskedasticity).")
print(bp_het_result.round(4).to_string(index=False))
if bp_het_pvalue < 0.05:
    print(" -> Reject H0: evidence of heteroskedasticity. Robust/clustered SEs "
          "(already used above) are appropriate.")
else:
    print(" -> Fail to reject H0: no strong evidence of heteroskedasticity.")

# --- 4.7c Breusch-Pagan LAGRANGE MULTIPLIER test for PANEL EFFECTS --------
# H0: Var(u_i) = 0, i.e. there is no firm-specific (entity) random effect
# and pooled OLS is an adequate specification. This is the classic
# Breusch & Pagan (1980) LM test for random effects, NOT a test of
# heteroskedasticity - it is testing whether a panel estimator (RE/FE) is
# needed at all, using the pooled OLS residuals arranged by firm and year.
#
# Test statistic (balanced or unbalanced panel):
#   LM = (N*Tbar) / (2*(Tbar-1)) * [ (sum_i (sum_t e_it)^2) / (sum_i sum_t e_it^2) - 1 ]^2 ~ chi2(1)
# where e_it are pooled OLS residuals, i indexes firms, t indexes years.
def breusch_pagan_lm_panel_effects_test(resid, entity_index):
    """Breusch & Pagan (1980) LM test for random/panel effects.

    resid         : pooled OLS residuals (array-like)
    entity_index   : matching entity (firm) identifier for each residual
    """
    resid = pd.Series(np.asarray(resid), index=entity_index)
    grouped = resid.groupby(level=0)
    sum_e_by_entity = grouped.sum()          # sum_t e_it, per firm
    n_t_by_entity = grouped.count()          # T_i, per firm
    N = n_t_by_entity.shape[0]                # number of firms
    total_obs = int(n_t_by_entity.sum())      # total firm-year observations
    T_bar = total_obs / N                     # average years per firm (handles unbalanced panels)

    numerator = (sum_e_by_entity ** 2).sum()
    denominator = (resid ** 2).sum()

    lm_stat = (N * T_bar) / (2 * (T_bar - 1)) * ((numerator / denominator) - 1) ** 2
    p_value = 1 - stats.chi2.cdf(lm_stat, df=1)
    return lm_stat, p_value, N, T_bar

entity_index_for_bp = reg_df.index.get_level_values("Company")
bp_lm_stat, bp_lm_pvalue, bp_lm_N, bp_lm_Tbar = breusch_pagan_lm_panel_effects_test(
    pooled_model.resid, entity_index_for_bp
)
bp_lm_result = pd.DataFrame({
    "Statistic": ["Breusch-Pagan LM (panel effects) chi2", "p-value", "N firms", "Avg. T per firm"],
    "Value": [bp_lm_stat, bp_lm_pvalue, bp_lm_N, bp_lm_Tbar],
})
print("\nTABLE 4.7c - Breusch-Pagan LM test for PANEL EFFECTS (Breusch & Pagan, 1980)")
print("H0: no firm-specific random effect (Var(u_i) = 0); pooled OLS is adequate.")
print(bp_lm_result.round(4).to_string(index=False))
if bp_lm_pvalue < 0.05:
    print(" -> Reject H0: significant panel effects present; pooled OLS is misspecified, "
          "a panel estimator (RE or FE) is preferred.")
else:
    print(" -> Fail to reject H0: no significant panel effects detected; "
          "pooled OLS may be adequate.")

# --- 4.7d Wooldridge test for serial correlation in panel data ------------
# H0: no first-order autocorrelation in the idiosyncratic error term.
# Procedure (Wooldridge, 2002; Drukker, 2003):
#   1. First-difference y and X within each firm: d y_it = y_it - y_i,t-1
#   2. Regress d y on d X (pooled OLS) and take the residuals, e_it_hat
#   3. Regress e_it_hat on its own first lag (by firm), clustering SEs by
#      firm; under the null of no serial correlation in the ORIGINAL levels
#      equation, the true slope coefficient in this auxiliary regression is
#      -0.5, so the test compares the estimated slope to -0.5 rather than
#      to 0.
def wooldridge_serial_correlation_test(panel_data, y_col, x_cols, entity_col_level=0, time_col_level=1):
    """Wooldridge (2002) test for serial correlation in a panel-data model.

    panel_data must be indexed by (entity, time) as in `panel_df`/`reg_df`.
    """
    d = panel_data.reset_index()
    entity_col = d.columns[entity_col_level]
    time_col = d.columns[time_col_level]
    d = d.sort_values([entity_col, time_col])

    diff_cols = []
    for col in [y_col] + x_cols:
        dcol = f"d_{col}"
        d[dcol] = d.groupby(entity_col)[col].diff()
        diff_cols.append(dcol)

    d_valid = d.dropna(subset=diff_cols).copy()

    y_diff = d_valid[f"d_{y_col}"]
    X_diff = sm.add_constant(d_valid[[f"d_{c}" for c in x_cols]])
    fd_model = sm.OLS(y_diff, X_diff).fit()

    d_valid["_resid"] = fd_model.resid.values
    d_valid["_resid_lag"] = d_valid.groupby(entity_col)["_resid"].shift(1)
    test_data = d_valid.dropna(subset=["_resid", "_resid_lag"]).copy()

    X_test = sm.add_constant(test_data["_resid_lag"])
    test_model = sm.OLS(test_data["_resid"], X_test).fit(
        cov_type="cluster", cov_kwds={"groups": test_data[entity_col]}
    )

    coef = test_model.params["_resid_lag"]
    se = test_model.bse["_resid_lag"]
    df_resid = int(test_model.df_resid)
    f_stat = ((coef - (-0.5)) / se) ** 2      # equivalent F(1, df_resid) test statistic
    f_pvalue = 1 - stats.f.cdf(f_stat, 1, df_resid)
    return coef, se, df_resid, f_stat, f_pvalue

wd_coef, wd_se, wd_df, wd_fstat, wd_pvalue = wooldridge_serial_correlation_test(
    reg_df, "ROA", ["ESG_Score", "FirmSize", "Leverage", "RevGrowth"]
)
wooldridge_result = pd.DataFrame({
    "Statistic": ["Lag coefficient (H0: = -0.5)", "Std. Error", "df", "F-stat", "p-value"],
    "Value": [wd_coef, wd_se, wd_df, wd_fstat, wd_pvalue],
})
print("\nTABLE 4.7d - Wooldridge test for SERIAL CORRELATION in panel data")
print("H0: no first-order autocorrelation in the idiosyncratic error term.")
print(wooldridge_result.round(4).to_string(index=False))
if wd_pvalue < 0.05:
    print(" -> Reject H0: evidence of first-order serial correlation. "
          "Clustered (by-firm) standard errors, as already used above, help "
          "account for this.")
else:
    print(" -> Fail to reject H0: no strong evidence of first-order serial correlation.")

# --- 4.7e Hausman test (Fixed Effects vs Random Effects) ---
def hausman_test(fe, re):
    """Manual Hausman specification test comparing FE and RE estimators."""
    common_vars = [v for v in fe.params.index if v in re.params.index]
    b_fe = fe.params[common_vars]
    b_re = re.params[common_vars]
    v_fe = fe.cov.loc[common_vars, common_vars]
    v_re = re.cov.loc[common_vars, common_vars]

    diff = b_fe - b_re
    var_diff = v_fe - v_re
    # Use pseudo-inverse for numerical stability
    stat = float(diff.T @ np.linalg.pinv(var_diff.values) @ diff)
    dof = len(common_vars)
    p_value = 1 - stats.chi2.cdf(stat, dof)
    return stat, dof, p_value

h_stat, h_dof, h_pvalue = hausman_test(fe_model, re_model)
print(f"\nTABLE 4.7e - Hausman test: chi2={h_stat:.4f}, df={h_dof}, p-value={h_pvalue:.4f}")
if h_pvalue < 0.05:
    print(" -> Reject H0: Fixed Effects model is preferred.")
else:
    print(" -> Fail to reject H0: Random Effects model is preferred.")

diagnostics_summary = pd.concat([
    vif_data.rename(columns={"Value": "Result"}),
    bp_het_result.rename(columns={"Test": "Variable", "Value": "Result"}),
    bp_lm_result.rename(columns={"Statistic": "Variable", "Value": "Result"}),
    wooldridge_result.rename(columns={"Statistic": "Variable", "Value": "Result"}),
    pd.DataFrame({"Variable": ["Hausman chi2", "Hausman df", "Hausman p-value"],
                  "Result": [h_stat, h_dof, h_pvalue]}),
], ignore_index=True)
diagnostics_summary.to_csv(os.path.join(OUT_DIR, "table_4_7_diagnostics.csv"), index=False)

# Also save each diagnostic test as its own CSV for clarity in the write-up
bp_het_result.to_csv(os.path.join(OUT_DIR, "table_4_7b_breusch_pagan_heteroskedasticity.csv"), index=False)
bp_lm_result.to_csv(os.path.join(OUT_DIR, "table_4_7c_breusch_pagan_LM_panel_effects.csv"), index=False)
wooldridge_result.to_csv(os.path.join(OUT_DIR, "table_4_7d_wooldridge_serial_correlation.csv"), index=False)

print("\n" + "=" * 80)
print("All tables and figures written to:", OUT_DIR)
print("=" * 80)

C:\Users\brian\anaconda3\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)



DATASET BUILT: 600 observations x 100 companies
Years covered: 2019 - 2024

TABLE 4.1 - Sample characteristics (first 15 firms shown)
                    Company                         Sector Currency     Years  Observations
                  88 Energy                  Oil & Gas E&P      USD 2019-2024             6
                 ADM Energy                  Oil & Gas E&P      USD 2019-2024             6
                 AFC Energy          Hydrogen & Fuel Cells      USD 2019-2024             6
                     Aminex                  Oil & Gas E&P      USD 2019-2024             6
               Angus Energy                  Oil & Gas E&P      USD 2019-2024             6
 Aquila European Renewables Renewable Energy (Diversified)      USD 2019-2024             6
          Arrow Exploration                  Oil & Gas E&P      USD 2019-2024             6
           Ascent Resources                  Oil & Gas E&P      USD 2019-2024             6
       Atrato Onsite Energy       Ren